Task A: Reward Model

Prasanna Paithankar (21CS30065)

Hugging Face Hub Link: [PrasannaPaithankar/reward_model_bert](https://huggingface.co/PrasannaPaithankar/bert-reward-model)

In [ ]:
import os

from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)
from trl.trainer.reward_config import RewardConfig
from trl.trainer.reward_trainer import RewardTrainer

load_dotenv()
login(token=os.getenv("HF_TOKEN"))

In [ ]:
ds = load_dataset("nrizwan/safe_ai_assignment_1")
ds["train"] = ds["train"].shuffle(seed=42).select(range(4000))


def preprocess_function(examples):
    return {
        "prompt": examples["Question"],
        "chosen": examples["More_Prefered"],
        "rejected": examples["Less_Prefered"],
    }


ds["train"] = ds["train"].map(preprocess_function, batched=True)
ds["test"] = ds["test"].map(preprocess_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.eos_token is None:
    tokenizer.eos_token = tokenizer.sep_token

training_args = RewardConfig(
    output_dir="./reward_model_bert",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_steps=1,
    disable_tqdm=False,
    num_train_epochs=4,
    remove_unused_columns=False,
    push_to_hub=True,
    hub_model_id="PrasannaPaithankar/bert-reward-model",
    max_length=512,
)

trainer = RewardTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 102}.


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,0.000010,0.011732
2,0.000010,0.010938
3,0.000006,0.010128


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1419, training_loss=0.017244137132646043, metrics={'train_runtime': 5749.5064, 'train_samples_per_second': 1.974, 'train_steps_per_second': 0.247, 'total_flos': 5792014129194300.0, 'train_loss': 0.017244137132646043})

In [6]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...el_bert/training_args.bin: 100%|##########| 5.46kB / 5.46kB            

  ...el_bert/model.safetensors:   7%|7         | 32.0MB /  438MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/PrasannaPaithankar/bert-reward-model/commit/98f7210aa6d11a31aeb25eb9d6d2167347ca5dc9', commit_message='End of training', commit_description='', oid='98f7210aa6d11a31aeb25eb9d6d2167347ca5dc9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/PrasannaPaithankar/bert-reward-model', endpoint='https://huggingface.co', repo_type='model', repo_id='PrasannaPaithankar/bert-reward-model'), pr_revision=None, pr_num=None)